## Stats Analysis

### Set up

### Base models

In [ ]:
import pandas as pd

RUNS = list(range(1, 6))
BASE_PATH = "../statistics"

INVERSION_THRESHOLD = 0.1
WASSERSTEIN_THRESHOLD = 0.15
RECONSTRUCTION_THRESHOLD = 2
CCC_THRESHOLD = 0.9

df_all = pd.read_csv(f"{BASE_PATH}/all_stats_r5_l16.csv")

df_test = df_all[df_all["key"] == "test_trans"]
df_open = df_all[df_all["key"].isin(["train_open", "train_closed"])]
df_closed = df_all[df_all["key"] == "train_closed"]


def style_table(df_split: pd.DataFrame, title: str):
    metric_cols = [
        "mean_rmsd",
        "std_rmsd",
        "inversion_ratio",
        "wd_N-CA",
        "ccc",
    ]
    target = df_split.set_index("var")[metric_cols]

    def highlight(val, col):
        if col == "mean_rmsd" and val > RECONSTRUCTION_THRESHOLD:
            return "background-color: #f4cccc"
        if col == "inversion_ratio" and val < INVERSION_THRESHOLD:
            return "background-color: #f4cccc"
        if col == "wd_N-CA" and val > WASSERSTEIN_THRESHOLD:
            return "background-color: #f4cccc"
        if col == "ccc" and val < CCC_THRESHOLD:
            return "background-color: #f4cccc"
        return ""

    styled = target.style.apply(lambda s: [highlight(v, s.name) for v in s], axis=0)
    styled = styled.set_caption(title).format("{:.4f}")
    return styled


numeric_metric_cols = ["mean_rmsd", "std_rmsd", "inversion_ratio", "wd_N-CA", "ccc"]

median_metrics_both = df_open.groupby("var")[numeric_metric_cols].median().reset_index()
median_metrics_test = df_test.groupby("var")[numeric_metric_cols].median().reset_index()

display(style_table(median_metrics_both, "Both"))
display(style_table(median_metrics_test, "Test"))

,mean_rmsd,std_rmsd,inversion_ratio,wd_N-CA,ccc
var,,,,,
2,1.1916,0.2021,0.0207,0.1336,0.7348
3,1.1489,0.1718,0.5353,0.0999,0.7760
4,1.1532,0.1460,0.5555,0.1085,0.8273
5,1.1511,0.1434,0.0847,0.1249,0.8664
6,1.2182,0.1605,0.0000,0.1623,0.8248
7,1.0089,0.1126,0.4010,0.0942,0.9071
8,1.0751,0.1189,0.0690,0.1031,0.8914
9,1.0061,0.1117,0.3628,0.1103,0.9032
10,0.9794,0.1023,0.2640,0.0943,0.9186


,mean_rmsd,std_rmsd,inversion_ratio,wd_N-CA,ccc
var,,,,,
2,2.2341,0.5634,0.0106,0.1486,0.8777
3,2.7687,1.2486,0.2712,0.1054,0.2871
4,1.9979,0.4051,0.0869,0.1283,0.9363
5,1.9278,0.3294,0.0106,0.1527,0.9622
6,1.8992,0.3532,0.0000,0.1778,0.9168
7,1.7668,0.3146,0.0512,0.1222,0.9668
8,1.9856,0.4365,0.0112,0.1242,0.9424
9,1.7720,0.3407,0.0625,0.1384,0.9668
10,1.6806,0.2730,0.0744,0.1187,0.9684


In [ ]:
# Aggregate metrics across runs using median for each latent dimension


### Line plots

In [ ]:
import matplotlib.pyplot as plt

metrics = [
    ("rmsd_mean", "rmsd_std", "Reconstruction RMSD"),
    ("inversion_mean", "inversion_std", "Inversion Ratio"),
    ("wd_mean", "wd_std", "Wasserstein Distance (N-CA)"),
    ("ccc_mean", "ccc_std", "Pairwise RMSD CCC"),
]

df_train = df_all[df_all["dataset"] == "train_both"].copy()
df_test = df_all[df_all["dataset"] == "test_trans"].copy()

fig, axes = plt.subplots(2, 2, figsize=(12, 8))

for ax, (mean_col, std_col, title) in zip(axes.flat, metrics):
    ax.errorbar(
        df_train["var"],
        df_train[mean_col],
        yerr=df_train[std_col],
        fmt="o-",
        capsize=3,
        label="Train",
    )
    ax.errorbar(
        df_test["var"],
        df_test[mean_col],
        yerr=df_test[std_col],
        fmt="s--",
        capsize=3,
        label="Test",
    )
    ax.set_xlabel("Latent Dimension")
    ax.set_ylabel(title)
    ax.set_title(title)
    ax.legend()

fig.tight_layout()
plt.show()